# 使用 OpenAI 的网站摘要器

一个有趣的网络爬虫小工具：抓取任意网站正文，再用 OpenAI API 生成尖刻、幽默的摘要。

## 这是做什么的

1. 从 URL 获取网站内容（HTTP 抓取）
2. 把正文发给 OpenAI 的 GPT 模型
3. 返回 Markdown 格式的尖刻摘要

非常适合快速了解一个网站在讲什么，而不必把整页废话读完。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `fetch_website_contents(url)`（来自 `scraper`） |
| Chat Completions | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定语气，user 塞网站正文 |
| Markdown 展示 | `display(Markdown(summary))` |


## 设置

确保仓库根目录有 `.env` 文件，并且包含：

```
OPENAI_API_KEY=your_key_here
```

然后按顺序从上到下运行下面的单元格（Shift+Enter）。


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从本目录 scraper 模块导入抓取函数：根据 URL 取网站正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI


## 连接到 OpenAI

这一步只做两件事：把 `.env` 读进环境变量，并检查 `OPENAI_API_KEY` 是否看起来合理。


In [ ]:
# ========== 环境：加载密钥并做简单格式校验 ==========

# 加载 .env；override=True 表示用文件里的值覆盖进程中已有的同名变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI API Key（常见名字：OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# 验证密钥：缺失 / 前缀不像 OpenAI project key / 看起来正常 —— 三种分支提示
if not api_key:
    print("Error: No API key found in .env")
elif not api_key.startswith("sk-proj-"):
    print("Warning: API key doesn't look like a valid OpenAI key")
else:
    print("API key loaded successfully!")


## 定义提示（Prompts）

- **system prompt**：告诉 GPT「你是谁、用什么语气、输出什么格式」
- **user prompt**：真正的任务说明；后面会再拼上网站正文

发给模型的英文指令字符串保持原样（改译会改变回答风格/行为）。


In [ ]:
# ========== system prompt：定 AI 性格与输出格式（发给模型的指令，不翻译） ==========

# 系统提示：尖酸幽默地总结网站；忽略导航类文字；用 Markdown，且不要包在代码块里
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [ ]:
# ========== user prompt 前缀：真正的任务说明（后面会 + 网站正文） ==========

# 用户提示前缀：要求短摘要；若有新闻/公告也一并概括（正文在 messages_for 里拼接）
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## 构建消息结构（messages）

OpenAI Chat Completions 需要一组带角色的消息：`system` / `user` /（可选）`assistant`。
下面的函数把「系统提示 + 用户提示前缀 + 网站正文」组装成 API 要求的列表。


In [ ]:
# ========== messages_for：把网站正文塞进 Chat Completions 的 messages 列表 ==========

def messages_for(website):
    """Create the message structure for OpenAI API"""
    # 返回两条消息：system 定风格；user = 前缀说明 + 抓到的网站文本
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


## 创建摘要函数

`summarize`：抓网页 → 调模型 → 返回字符串。  
`display_summary`：再把结果用 Markdown 漂亮显示出来。


In [ ]:
# ========== summarize：抓取网页并用 GPT 生成尖刻摘要 ==========

def summarize(url):
    """Fetch a website and return a snarky summary"""
    # 创建 OpenAI 客户端（默认从环境变量读 OPENAI_API_KEY）
    openai = OpenAI()
    # 抓取 URL 对应网站的正文文本
    website = fetch_website_contents(url)
    # 调用 Chat Completions：指定小模型 + 组装好的 messages
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for(website)
    )
    # 取出第一条 choice 里 assistant 的文本内容并返回
    return response.choices[0].message.content


In [ ]:
# ========== display_summary：调用 summarize，并在笔记本里用 Markdown 展示 ==========

def display_summary(url):
    """Display the summary as formatted markdown"""
    # 先拿到摘要字符串
    summary = summarize(url)
    # 用 IPython 的 Markdown 渲染显示
    display(Markdown(summary))


## 尝试一下！

改下面单元格里的 URL，就能摘要不同网站。第一次建议先跑作者主页示例。


In [ ]:
# ========== 试跑：摘要一个相对简单的静态站点 ==========

# 尝试一个简单的网站（URL 字符串保持原样，影响抓取目标）
display_summary("https://edwarddonner.com")


In [ ]:
# ========== 再试：换一个更短的示例站 ==========

# 尝试更多网站（可改成你想摘要的 URL）
display_summary("https://example.com")


## 注意事项

- 这最适合**静态 HTML**网站
- 靠 JavaScript 渲染的网站（React 应用、SPA）往往抓不到真正内容
- 某些有 CDN / 反爬保护的站点可能返回 403
- 若要抓动态站，可参考社区文件夹里的 Selenium 或 Playwright 实现

## 定制语气

想改变语气吗？编辑上面的 `system_prompt` 变量，例如：

- 去掉「尖酸刻薄」，改成专业摘要
- 加上「请用中文回答」之类的语言说明
- 调整成正式 / 技术向 / 更有趣等不同风格
